In [28]:
%pip -q install tensorflow
import os
import re
import zipfile
import numpy as np
import tensorflow as tf

from google.colab import files
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [29]:
zip_path = "/content/archive (1).zip"

if not os.path.exists(zip_path):
    uploaded = files.upload()

    zip_files = [
        name for name in uploaded.keys()
        if name.lower().endswith(".zip")
    ]

    if not zip_files:
        raise FileNotFoundError(
            "Please upload the provided ZIP dataset."
        )

    zip_path = os.path.join("/content", zip_files[0])

extract_dir = "/content/next_word_dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

txt_files = []

for root, dirs, filenames in os.walk(extract_dir):
    for filename in filenames:
        if filename.lower().endswith(".txt"):
            txt_files.append(
                os.path.join(root, filename)
            )

if not txt_files:
    raise FileNotFoundError(
        "No .txt dataset file found inside ZIP."
    )

txt_path = txt_files[0]

print("Dataset file:", txt_path)

Dataset file: /content/next_word_dataset/1661-0.txt


In [30]:
with open(
    txt_path,
    "r",
    encoding="utf-8-sig",
    errors="ignore"
) as file:
    raw_text = file.read()

start_marker = "*** START OF"
end_marker = "*** END OF"

start_match = re.search(
    re.escape(start_marker),
    raw_text,
    flags=re.IGNORECASE
)

if start_match:
    raw_text = raw_text[start_match.end():]

    first_newline = raw_text.find("\n")

    if first_newline != -1:
        raw_text = raw_text[first_newline + 1:]

end_match = re.search(
    re.escape(end_marker),
    raw_text,
    flags=re.IGNORECASE
)

if end_match:
    raw_text = raw_text[:end_match.start()]

raw_text = raw_text.lower()

sentences = re.split(
    r"[.!?;:\n]+",
    raw_text
)

clean_sentences = []

for sentence in sentences:
    words = re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        sentence
    )

    if len(words) >= 3:
        clean_sentences.append(" ".join(words))

print("Number of usable sentences:", len(clean_sentences))

print(
    "Example sentence:",
    clean_sentences[0][:250]
    if clean_sentences else "No text found"
)

Number of usable sentences: 11597
Example sentence: produced by an anonymous project gutenberg volunteer and jose menendez


In [31]:
MAX_VOCAB = 5000
MAX_SEQUENCE_LENGTH = 15

# Split sentences before creating sequences
train_sentences, test_sentences = train_test_split(
    clean_sentences,
    test_size=0.15,
    random_state=42
)

tokenizer = Tokenizer(
    num_words=MAX_VOCAB,
    oov_token="<OOV>"
)

# Fit tokenizer only on training sentences
tokenizer.fit_on_texts(train_sentences)

train_sequences = tokenizer.texts_to_sequences(
    train_sentences
)

test_sequences = tokenizer.texts_to_sequences(
    test_sentences
)

vocab_size = min(
    MAX_VOCAB,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size:", vocab_size)
print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))

Vocabulary size: 5000
Training sentences: 9857
Testing sentences: 1740


In [32]:
def create_sequences(sentences, tokenizer):

    sequences = []

    for sentence in sentences:

        word_ids = tokenizer.texts_to_sequences(
            [sentence]
        )[0]

        for i in range(1, len(word_ids)):

            context = word_ids[
                max(0, i - MAX_SEQUENCE_LENGTH):i
            ]

            target = word_ids[i]

            # Ignore unknown target words
            if target > 1:

                sequences.append(
                    (context, target)
                )

    return sequences


train_data = create_sequences(
    train_sentences,
    tokenizer
)

test_data = create_sequences(
    test_sentences,
    tokenizer
)

print("Training sequences:", len(train_data))
print("Testing sequences:", len(test_data))

Training sequences: 75191
Testing sequences: 12788


In [33]:
X_train = [item[0] for item in train_data]
y_train = np.array(
    [item[1] for item in train_data]
)

X_test = [item[0] for item in test_data]
y_test = np.array(
    [item[1] for item in test_data]
)

X_train = pad_sequences(
    X_train,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="pre",
    truncating="pre"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="pre",
    truncating="pre"
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (75191, 15)
y_train shape: (75191,)
X_test shape: (12788, 15)
y_test shape: (12788,)


In [34]:
tf.keras.backend.clear_session()

tf.keras.utils.set_random_seed(42)

model = models.Sequential([

    layers.Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
        mask_zero=True
    ),

    layers.LSTM(
        128,
        return_sequences=True
    ),

    layers.Dropout(0.2),

    layers.LSTM(64),

    layers.Dropout(0.2),

    layers.Dense(
        vocab_size,
        activation="softmax"
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy"
        )
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 15, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 15, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5000)           │       325,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,145,992 (4.37 MB)

 Trainable params: 1,145,992 (4.37 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

history = model.fit(
    X_train,
    y_train,

    validation_split=0.10,

    epochs=30,

    batch_size=64,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/30
1058/1058 ━━━━━━━━━━━━━━━━━━━━ 106s 91ms/step - accuracy: 0.0550 - loss: 6.5728 - top_5_accuracy: 0.1667 - val_accuracy: 0.0574 - val_loss: 6.2893 - val_top_5_accuracy: 0.1649 - learning_rate: 0.0010
Epoch 2/30
1058/1058 ━━━━━━━━━━━━━━━━━━━━ 93s 88ms/step - accuracy: 0.0558 - loss: 6.3283 - top_5_accuracy: 0.1681 - val_accuracy: 0.0574 - val_loss: 6.2086 - val_top_5_accuracy: 0.1672 - learning_rate: 0.0010
Epoch 3/30
1058/1058 ━━━━━━━━━━━━━━━━━━━━ 141s 87ms/step - accuracy: 0.0583 - loss: 6.1287 - top_5_accuracy: 0.1750 - val_accuracy: 0.0723 - val_loss: 6.0119 - val_top_5_accuracy: 0.1848 - learning_rate: 0.0010
Epoch 4/30
1058/1058 ━━━━━━━━━━━━━━━━━━━━ 143s 88ms/step - accuracy: 0.0753 - loss: 5.8989 - top_5_accuracy: 0.2045 - val_accuracy: 0.0924 - val_loss: 5.7882 - val_top_5_accuracy: 0.2362 - learning_rate: 0.0010
Epoch 5/30
1058/1058 ━━━━━━━━━━━━━━━━━━━━ 96s 90ms/step - accuracy: 0.1002 - loss: 5.6416 - top_5_accuracy: 0.2587 - val_accuracy: 0.1158 - val_loss: 5.5822 

In [36]:
results = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nModel Evaluation:")

for name, value in zip(
    model.metrics_names,
    results
):
    print(f"{name}: {value:.4f}")

400/400 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.1419 - loss: 5.1403 - top_5_accuracy: 0.3204

Model Evaluation:
loss: 5.1403
compile_metrics: 0.1419


In [37]:
def predict_next_word(input_text, top_k=5):

    words = re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        input_text.lower()
    )

    if not words:
        print("Please enter a valid sentence.")
        return

    sequence = tokenizer.texts_to_sequences(
        [" ".join(words)]
    )[0]

    sequence = sequence[-MAX_SEQUENCE_LENGTH:]

    padded = pad_sequences(
        [sequence],
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="pre"
    )

    predictions = model.predict(
        padded,
        verbose=0
    )[0]

    top_indices = np.argsort(
        predictions
    )[::-1]

    print("\nInput:", input_text)
    print("\nNext Word Predictions:")

    count = 0

    for index in top_indices:

        word = tokenizer.index_word.get(
            int(index)
        )

        if word and word != "<OOV>":

            print(
                f"{word}: {predictions[index]:.4f}"
            )

            count += 1

        if count == top_k:
            break

In [38]:
test_sentences = [
    "sherlock holmes was",
    "the man looked at",
    "i have never seen",
    "it was a very",
    "he said that",
    "the detective entered the"
]

for sentence in test_sentences:

    print("\n" + "=" * 50)

    predict_next_word(
        sentence,
        top_k=5
    )



Input: sherlock holmes was

Next Word Predictions:
a: 0.0848
the: 0.0232
in: 0.0227
very: 0.0171
not: 0.0136


Input: the man looked at

Next Word Predictions:
the: 0.4784
his: 0.0866
a: 0.0800
my: 0.0368
him: 0.0148


Input: i have never seen

Next Word Predictions:
a: 0.1220
the: 0.0969
to: 0.0532
in: 0.0497
him: 0.0274


Input: it was a very

Next Word Predictions:
little: 0.0124
man: 0.0100
interesting: 0.0095
few: 0.0073
household: 0.0066


Input: he said that

Next Word Predictions:
i: 0.2479
he: 0.1000
you: 0.0854
the: 0.0686
she: 0.0443


Input: the detective entered the

Next Word Predictions:
little: 0.0183
man: 0.0139
matter: 0.0136
door: 0.0123
house: 0.0092


In [39]:


test_loss, test_accuracy, test_top5_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\n========== MODEL PERFORMANCE ==========")

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

print(f"Test Loss: {test_loss:.4f}")

print(f"Top-5 Accuracy: {test_top5_accuracy * 100:.2f}%")

print("=======================================")

400/400 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.1419 - loss: 5.1403 - top_5_accuracy: 0.3204

========== MODEL PERFORMANCE ==========
Test Accuracy: 14.19%
Test Loss: 5.1403
Top-5 Accuracy: 32.04%


In [40]:
print("FINAL TRAINING ACCURACY:",
      history.history["accuracy"][-1] * 100, "%")

print("FINAL VALIDATION ACCURACY:",
      history.history["val_accuracy"][-1] * 100, "%")

print("FINAL TRAINING LOSS:",
      history.history["loss"][-1])

print("FINAL VALIDATION LOSS:",
      history.history["val_loss"][-1])

FINAL TRAINING ACCURACY: 18.05795729160309 %
FINAL VALIDATION ACCURACY: 14.760638773441315 %
FINAL TRAINING LOSS: 4.52963924407959
FINAL VALIDATION LOSS: 5.365820407867432


In [41]:
print("PROJECT: NEXT WORD PREDICTION USING LSTM")

print("Dataset: Sherlock Holmes Text")

print("Model: LSTM Neural Network")

print("Training Accuracy: 18.06%")

print("Validation Accuracy: 14.76%")

print("Training completed successfully.")

print("Next word prediction function implemented.")

PROJECT: NEXT WORD PREDICTION USING LSTM
Dataset: Sherlock Holmes Text
Model: LSTM Neural Network
Training Accuracy: 18.06%
Validation Accuracy: 14.76%
Training completed successfully.
Next word prediction function implemented.
